# RAG Seguridad Pública México — Google Colab

**Pipeline completo**: 23 PDFs → 7595 chunks → ChromaDB → Qwen 0.5B → LoRA → Chat con fuentes

**Proyecto 4 de IA** · jojo · Repo: [github.com/JoelJohs/repo_de_IA](https://github.com/JoelJohs/repo_de_IA)

---

## ⏱️ Tiempos esperados en Colab (T4)

| Etapa | Primera vez | Sesión siguiente |
|---|---:|---:|
| Setup (Drive + pip + GPU) | 2-3 min | 30 s |
| Descargar 23 PDFs | 5-10 min | 0 (en Drive) |
| Chunking | 1-2 min | 0 (en Drive) |
| Embeddings + ChromaDB | 5-10 min | 30 s (cargar) |
| Fine-tuning LoRA (3 epochs) | **6-10 min** | 0 (en Drive) |
| Query de prueba | 3-10 s | 3-10 s |
| **TOTAL primera vez** | **~25-40 min** | **30 s por query** |

**Espacio en Drive necesario**: ~3 GB (corpus 184 MB + vectordb 74 MB + LoRA 8 MB + cache modelos ~2 GB)


## 📋 Cómo usar este notebook

1. **Abrir en Colab**: `Archivo → Subir notebook` o arrastrar el `.ipynb`
2. **Activar GPU**: `Entorno de ejecución → Cambiar tipo → GPU T4`
3. **Ejecutar celdas en orden**, de arriba a abajo (botón ▶️ o `Shift+Enter`)
4. **Si la sesión se corta**: Drive persiste todo, solo reejecutar desde donde se cortó

**Tolerancia a fallos**:
- Las celdas verifican si el output ya existe en Drive y lo saltean
- Si una celda falla, podés reejecutarla sin rehacer el resto
- Drive se puede re-montar con `force_remount=True`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  NO hay GPU. El LoRA va a tardar ~1 hora/epoch en vez de 2-3 min.")
    print("   Cambiar tipo de entorno: Entorno de ejecución → Cambiar tipo → GPU T4")


In [ ]:
!pip install -q \
  langchain>=0.3.0 \
  langchain-community>=0.3.0 \
  langchain-huggingface>=1.0.0 \
  langchain-chroma>=1.0.0 \
  langchain-core>=0.3.0 \
  langchain-text-splitters>=0.3.0 \
  chromadb>=0.5.0 \
  sentence-transformers>=3.0.0 \
  pypdf>=5.0.0 \
  PyMuPDF>=1.27.0 \
  peft>=0.12.0 \
  trl>=0.9.0 \
  bitsandbytes>=0.43.0 \
  accelerate>=0.33.0 \
  datasets>=2.20.0 \
  transformers>=4.44.0 \
  gradio>=4.0.0

print("\n✓ Dependencias instaladas")

# Si bitsandbytes falla, descomentar y reejecutar:
# !pip install -q bitsandbytes --no-build-isolation


In [ ]:
import os
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/4-RAG-colab")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

CORPUS_DIR    = DRIVE_ROOT / "corpus"
VECTORDB_DIR  = DRIVE_ROOT / "vectordb"
LORA_DIR      = DRIVE_ROOT / "lora"
HF_CACHE      = DRIVE_ROOT / "hf_cache"
CHUNKS_FILE   = DRIVE_ROOT / "chunks.jsonl"
FINETUNE_FILE = DRIVE_ROOT / "finetuning.jsonl"
DOCS_FILE     = DRIVE_ROOT / "documentos.json"

for d in [CORPUS_DIR, VECTORDB_DIR, LORA_DIR, HF_CACHE]:
    d.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE)
os.environ["SENTENCE_TRANSFORMERS_HOME"] = str(HF_CACHE)

print(f"Drive root: {DRIVE_ROOT}")
print(f"Subdirs: corpus, vectordb, lora, hf_cache")


In [ ]:
import json
import requests

DOCS_URL = "https://raw.githubusercontent.com/JoelJohs/repo_de_IA/main/4-RAG/corpus/pdf/metadata/documentos.json"
print(f"Descargando catalogo: {DOCS_URL}")

r = requests.get(DOCS_URL)
r.raise_for_status()
documentos = r.json()

with DOCS_FILE.open("w") as f:
    json.dump(documentos, f, indent=2, ensure_ascii=False)

print(f"Documentos: {len(documentos)}")
print(f"Categorias unicas: {len(set(d['categoria'] for d in documentos))}")
print(f"Instituciones unicas: {len(set(d['institucion'] for d in documentos))}")
print(f"\nMuestra del primer documento:")
print(json.dumps(documentos[0], indent=2, ensure_ascii=False))


In [ ]:
import requests
from pathlib import Path

def download_pdf(url, dest):
    if dest.exists() and dest.stat().st_size > 1000:
        return True  # ya bajado
    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=120)
        r.raise_for_status()
        if r.content[:4] != b"%PDF":
            print(f"  ✗ No es PDF: {url[:80]}")
            return False
        dest.write_bytes(r.content)
        return True
    except Exception as e:
        print(f"  ✗ Error: {url[:80]}: {e}")
        return False


ok, fail, skip = 0, 0, 0
for doc in documentos:
    url = doc.get("url")
    cat = doc.get("categoria", "misc")
    inst = doc.get("institucion", "unknown").lower().replace(" ", "_")
    anio = doc.get("anio", "0000")
    titulo = doc.get("titulo", "doc")[:40].replace(" ", "_").replace("/", "_")
    fname = f"{cat}_{inst}_{anio}_{titulo}.pdf"
    dest = CORPUS_DIR / cat / fname
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 1000:
        skip += 1
        continue
    if download_pdf(url, dest):
        ok += 1
        print(f"  ✓ {fname[:80]}")
    else:
        fail += 1

total_pdfs = len(list(CORPUS_DIR.rglob("*.pdf")))
print(f"\n=== Resumen ===")
print(f"Descargados nuevos: {ok}")
print(f"Ya existían: {skip}")
print(f"Fallos: {fail}")
print(f"Total PDFs en corpus: {total_pdfs}")

if total_pdfs < 20:
    print("\n⚠️  Hay menos de 20 PDFs. Puede haber fallado la descarga.")
    print("    Razon probable: la institucion bloqueo la IP de Colab.")
    print("    Solucion: subir un zip de los PDFs a Drive manualmente.")


In [ ]:
import json
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
SEPARATORS = ["\n\n", "\n", ". ", ", ", " ", ""]

if CHUNKS_FILE.exists():
    n_existing = sum(1 for _ in CHUNKS_FILE.open())
    print(f"✓ chunks.jsonl ya existe en Drive ({n_existing} chunks). Saltando chunking.")
    print("  Para forzar re-chunking: borrar el archivo y reejecutar esta celda.")
else:
    docs_meta = {}
    for doc in documentos:
        cat = doc.get("categoria", "misc")
        inst = doc.get("institucion", "unknown").lower().replace(" ", "_")
        anio = doc.get("anio", "0000")
        titulo = doc.get("titulo", "doc")[:40].replace(" ", "_").replace("/", "_")
        fname = f"{cat}_{inst}_{anio}_{titulo}.pdf"
        docs_meta[fname] = {
            "author": doc.get("autor", "?"),
            "institution": doc.get("institucion", "?"),
            "year": doc.get("anio", "?"),
            "category": cat,
        }

    pdfs = list(CORPUS_DIR.rglob("*.pdf"))
    print(f"PDFs a procesar: {len(pdfs)}")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=SEPARATORS,
    )

    chunks = []
    for pdf_path in pdfs:
        categoria = pdf_path.parent.name
        fname = pdf_path.name
        enriched = docs_meta.get(fname, {
            "author": "?", "institution": "?", "year": "?", "category": categoria,
        })
        try:
            pages = PyPDFLoader(str(pdf_path)).load()
        except Exception as e:
            print(f"  ✗ Error leyendo {fname}: {e}")
            continue
        if not pages:
            continue
        docs = splitter.split_documents(pages)
        for idx, doc in enumerate(docs):
            text = doc.page_content.strip()
            if not text:
                continue
            chunks.append({
                "chunk_id": f"{pdf_path.stem}_{idx:04d}",
                "text": text,
                "metadata": {
                    **enriched,
                    "source_file": str(pdf_path.relative_to(CORPUS_DIR)),
                    "pages": str(doc.metadata.get("page", "")),
                    "chunk_index": idx,
                },
            })

    with CHUNKS_FILE.open("w") as f:
        for c in chunks:
            f.write(json.dumps(c, ensure_ascii=False) + "\n")

    print(f"\n✓ Total chunks: {len(chunks)} -> {CHUNKS_FILE}")
    print(f"\nMuestra de metadata del primer chunk:")
    print(json.dumps(chunks[0]["metadata"], indent=2, ensure_ascii=False))


In [ ]:
import json
import shutil
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

if VECTORDB_DIR.exists() and (VECTORDB_DIR / "chroma.sqlite3").exists():
    print(f"✓ vectordb ya existe en {VECTORDB_DIR}. Cargando...")
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )
    vectordb = Chroma(
        persist_directory=str(VECTORDB_DIR),
        embedding_function=embeddings,
        collection_name="seguridad_mexico",
    )
else:
    print("Construyendo ChromaDB desde cero (~5-10 min)...")
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )

    docs = []
    with CHUNKS_FILE.open() as f:
        for line in f:
            c = json.loads(line)
            meta = c["metadata"].copy()
            meta.pop("chunk_index", None)
            docs.append(Document(page_content=c["text"], metadata=meta, id=c["chunk_id"]))

    print(f"  Documentos a indexar: {len(docs)}")

    if VECTORDB_DIR.exists():
        shutil.rmtree(VECTORDB_DIR)

    vectordb = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        persist_directory=str(VECTORDB_DIR),
        collection_name="seguridad_mexico",
        collection_metadata={"hnsw:space": "cosine"},
    )
    print(f"  ✓ ChromaDB creada")

n_chunks = vectordb._collection.count()
print(f"\n=== Vectordb: {n_chunks} chunks en {VECTORDB_DIR} ===")
size_mb = sum(f.stat().st_size for f in VECTORDB_DIR.rglob("*") if f.is_file()) / 1e6
print(f"Tamaño: {size_mb:.1f} MB")


In [ ]:
MANIFIESTO = """\
# Sobre este corpus y sus autores

## El curador

Este corpus fue **reunido por jojo** como parte del Proyecto 4 de la
materia de Inteligencia Artificial. jojo es estudiante de ingeniería,
no es experto en seguridad pública. Su rol fue **seleccionar y
organizar** documentos institucionales relevantes, no producir análisis
propio.

Si te preguntan "¿quién es el autor del corpus?" o "¿quién hizo este
sistema?", la respuesta es: **jojo**, como curador. jojo no tiene
postura editorial; organizó documentos de muchas instituciones.

## Los autores del corpus

El corpus **no tiene una postura unificada**. Contiene 23 documentos
de 16 instituciones diferentes, publicadas entre 2021 y 2025. Las
principales voces se agrupan así:

- **Fuentes oficiales** (cifras, estadísticas, marco legal): INEGI,
  Cámara de Diputados, World Bank, ICRC, U.S. Department of State.
- **Think tanks y observatorios críticos**: México Evalúa, ONC, MUCD,
  Integralia, International Crisis Group, Noria Research.
- **Académicas**: UNAM, Fundación Carolina.
- **Derechos humanos y humanitarias**: Centro Prodh, SCJN/ACNUR/CICR,
  UNODC.

## Cómo responder sobre "el autor"

Cuando te pregunten **"¿qué piensa el autor del corpus?"**, hay tres
interpretaciones posibles. La convención de este sistema es:

### 1. Si preguntan por el curador
"Este corpus fue curado por **jojo**, estudiante de la materia de
Inteligencia Artificial, como parte del Proyecto 4. jojo organizó 23
documentos de 16 instituciones; su rol fue seleccionar y clasificar,
no producir análisis editorial. Los autores del contenido son las
instituciones listadas en AUTORES.md."

### 2. Si preguntan por un autor institucional específico
"Según {institución} en {título del documento}, ..." y citar el
contexto recuperado. El RAG filtra por metadata `institution` cuando
la pregunta lo permite.

### 3. Si preguntan por la postura general del corpus
"El corpus es plural: no hay una postura única. Sobre {tema}, las
instituciones tienen perspectivas que van desde {voz oficialista}
hasta {voz crítica}. El contexto recuperado contiene los argumentos
de cada una."

## Sobre temas con disenso

En temas como **cifra negra**, **militarización** y **subregistro**,
las instituciones del corpus tienen **posturas divergentes**. La
convención es:

- Si hay cifras oficiales, priorizarlas (ej. INEGI).
- Si hay crítica metodológica, presentarla (ej. México Evalúa).
- Si hay ONGs cuestionando al Estado, dar voz a esas ONGs (ej. Centro
  Prodh, MUCD).
- **No sintetizar artificialmente**: si hay desacuerdo, mostrarlo.

## Lo que este sistema NO hace

- No produce postura editorial propia. jojo como curador no opina; las
  posturas son de los documentos.
- No oculta fuentes. Cada afirmación va atribuida a su institución.
- No alucina datos. Si el contexto no contiene la respuesta, lo dice.
"""

print(f"Manifiesto: {len(MANIFIESTO)} caracteres")


def format_docs_with_attribution(docs):
    """Formatea los chunks retrieved con atribucion explicita."""
    parts = []
    for i, d in enumerate(docs, 1):
        m = d.metadata
        inst = m.get("institution", m.get("institucion", "?"))
        author = m.get("author", m.get("autor", "?"))
        year = m.get("year", m.get("anio", "?"))
        src = m.get("source_file", m.get("source", "?"))
        pages = m.get("pages", "?")
        header = f"[Fuente {i}] {inst} / {author} ({year}) — {src} (pag. {pages})"
        parts.append(f"{header}\n{d.page_content}")
    return "\n\n".join(parts)


def build_rag_prompt(context, question, include_manifesto=True):
    """Prompt completo: MANIFIESTO + contexto + pregunta."""
    manifesto_section = f"{MANIFIESTO}\n\n" if include_manifesto else ""
    return (
        f"{manifesto_section}"
        "Eres un asistente experto en seguridad publica en Mexico. "
        "Responde SOLO con la informacion del contexto proporcionado. "
        "Cuando el contexto incluya datos cuantitativos, cita la institucion y el titulo. "
        "Si hay disenso entre autores, presenta las multiples perspectivas. "
        "Si el contexto no contiene la informacion, indicálo explicitamente.\n\n"
        f"Contexto:\n{context}\n\n"
        f"Pregunta: {question}\n\n"
        "Respuesta (en espanol, basada unicamente en el contexto):"
    )


# Test rapido
test_docs = vectordb.as_retriever(search_kwargs={"k": 3}).invoke("homicidios")
print(f"Test retrieval: {len(test_docs)} chunks para 'homicidios'")
print(f"\nMuestra de formato con atribucion:")
print(format_docs_with_attribution(test_docs[:1])[:400])


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
print(f"Cargando {BASE_MODEL}...")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",
    trust_remote_code=True,
)
base_model.eval()
print(f"✓ Modelo en {next(base_model.parameters()).device}")
print(f"  VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


def generate_base(prompt, max_new_tokens=256, temperature=0.1):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to("cuda") for k, v in inputs.items()}
    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "<|assistant|>" in answer:
        answer = answer.split("<|assistant|>")[-1].strip()
    return answer


# Test rapido del modelo
test = """<|system|>\nSos un asistente util.</s>\n<|user|>\nDi hola.</s>\n<|assistant|>"""
print(f"\nTest: {generate_base(test, max_new_tokens=30)}")


In [ ]:
import time

def rag_query_base(question, k=5, max_new_tokens=300):
    retriever = vectordb.as_retriever(search_kwargs={"k": k})
    t0 = time.time()
    docs = retriever.invoke(question)
    retrieve_time = time.time() - t0

    context = format_docs_with_attribution(docs)
    prompt = build_rag_prompt(context, question, include_manifesto=True)

    t0 = time.time()
    answer = generate_base(prompt, max_new_tokens=max_new_tokens, temperature=0.1)
    gen_time = time.time() - t0

    return {
        "answer": answer,
        "sources": [
            {"content": d.page_content[:200] + "...", "metadata": d.metadata}
            for d in docs
        ],
        "timing": {"retrieve_s": round(retrieve_time, 2), "generate_s": round(gen_time, 2)},
    }


print("=" * 60)
print("TEST: RAG base (sin LoRA)")
print("=" * 60)
result = rag_query_base("¿Cuántos homicidios hubo en México en 2024?")
print(f"\nRESPUESTA:\n{result['answer']}")
print(f"\nFUENTES ({len(result['sources'])}):")
for i, src in enumerate(result['sources'], 1):
    m = src['metadata']
    print(f"  [{i}] {m.get('institution', '?')} ({m.get('year', '?')}) — {m.get('source_file', '?')}")
print(f"\nTIEMPO: {result['timing']}")


In [ ]:
import requests
import json

FINETUNING_URL = "https://raw.githubusercontent.com/JoelJohs/repo_de_IA/main/4-RAG/datasets/finetuning.jsonl"
print(f"Descargando {FINETUNING_URL}...")

r = requests.get(FINETUNING_URL)
r.raise_for_status()

with FINETUNE_FILE.open("w") as f:
    f.write(r.text)

records = [json.loads(l) for l in FINETUNE_FILE.read_text().splitlines() if l.strip()]
print(f"\n✓ {len(records)} ejemplos cargados")

from collections import Counter
behaviors = Counter(r.get("behavior", "?") for r in records)
print(f"\nDistribucion por comportamiento:")
for b, c in behaviors.most_common():
    print(f"  {b}: {c}")

print(f"\nMuestra del primer ejemplo:")
print(json.dumps(records[0], indent=2, ensure_ascii=False)[:500])


In [ ]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

if (LORA_DIR / "adapter_config.json").exists():
    print(f"✓ Adapter ya existe en {LORA_DIR}. Saltando training.")
    print("  Para reentrenar: borrar el dir y reejecutar esta celda.")
else:
    print("Preparando QLoRA 4-bit...")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    train_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    if train_tokenizer.pad_token is None:
        train_tokenizer.pad_token = train_tokenizer.eos_token

    train_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="cuda:0",
        trust_remote_code=True,
    )
    train_model = prepare_model_for_kbit_training(train_model)

    lora_config = LoraConfig(
        r=8,
        lora_alpha=32,
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    train_model = get_peft_model(train_model, lora_config)
    train_model.print_trainable_parameters()

    def format_for_training(record):
        instruction = record.get("instruction", "")
        response = record.get("response") or record.get("output", "")
        return (
            f"<|system|>\nEres un asistente experto en seguridad publica en Mexico. "
            f"Responde en espanol.</s>\n"
            f"<|user|>\n{instruction}</s>\n"
            f"<|assistant|>\n{response}</s>"
        )

    texts = [format_for_training(r) for r in records]
    dataset = Dataset.from_list([{"text": t} for t in texts])

    def tokenize_fn(examples):
        return train_tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

    tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

    training_args = TrainingArguments(
        output_dir=str(LORA_DIR / "checkpoints"),
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        warmup_steps=10,
        logging_steps=5,
        save_steps=50,
        save_total_limit=2,
        fp16=False,
        bf16=True,
        dataloader_pin_memory=False,
        report_to="none",
    )

    trainer = Trainer(
        model=train_model,
        args=training_args,
        train_dataset=tokenized,
        tokenizer=train_tokenizer,
    )

    print(f"\nIniciando training: 3 epochs sobre {len(records)} ejemplos...")
    t0 = time.time()
    trainer.train()
    print(f"\n✓ Training completo en {(time.time()-t0)/60:.1f} min")

    trainer.save_model(str(LORA_DIR))
    train_tokenizer.save_pretrained(str(LORA_DIR))
    print(f"✓ Adapter guardado en {LORA_DIR}")

    del train_model, trainer
    torch.cuda.empty_cache()


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

if not (LORA_DIR / "adapter_config.json").exists():
    print("⚠️  No hay adapter. Ejecutá la celda anterior primero.")
else:
    print("Recargando modelo base + LoRA adapter...")
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.bfloat16,
        device_map="cuda:0",
        trust_remote_code=True,
    )
    lora_model = PeftModel.from_pretrained(base, str(LORA_DIR))
    lora_model.eval()
    print(f"✓ Modelo + LoRA en {next(lora_model.parameters()).device}")
    print(f"  VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

    def generate_lora(prompt, max_new_tokens=300, temperature=0.1):
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
        inputs = {k: v.to("cuda") for k, v in inputs.items()}
        with torch.no_grad():
            outputs = lora_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
            )
        answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
        if "<|assistant|>" in answer:
            answer = answer.split("<|assistant|>")[-1].strip()
        return answer


In [ ]:
def rag_query_lora(question, k=5, max_new_tokens=300):
    retriever = vectordb.as_retriever(search_kwargs={"k": k})
    t0 = time.time()
    docs = retriever.invoke(question)
    retrieve_time = time.time() - t0

    context = format_docs_with_attribution(docs)
    prompt = build_rag_prompt(context, question, include_manifesto=True)

    t0 = time.time()
    answer = generate_lora(prompt, max_new_tokens=max_new_tokens, temperature=0.1)
    gen_time = time.time() - t0

    return {
        "answer": answer,
        "sources": [
            {"content": d.page_content[:200] + "...", "metadata": d.metadata}
            for d in docs
        ],
        "timing": {"retrieve_s": round(retrieve_time, 2), "generate_s": round(gen_time, 2)},
    }


print("=" * 60)
print("TEST: RAG + LoRA")
print("=" * 60)
result = rag_query_lora("¿Quién es el autor del corpus?")
print(f"\nRESPUESTA:\n{result['answer']}")
print(f"\nFUENTES ({len(result['sources'])}):")
for i, src in enumerate(result['sources'], 1):
    m = src['metadata']
    print(f"  [{i}] {m.get('institution', '?')} ({m.get('year', '?')}) — {m.get('source_file', '?')}")
print(f"\nTIEMPO: {result['timing']}")


In [ ]:
test_questions = [
    "¿Cuántos homicidios hubo en México en 2024?",
    "¿Quién es el autor del corpus?",
    "¿Qué instituciones están representadas?",
    "¿Tiene el corpus una postura editorial?",
    "¿Cuál es la cifra negra de criminalidad en México?",
]

print("=" * 70)
print("COMPARATIVA: RAG base vs RAG + LoRA")
print("=" * 70)
for q in test_questions:
    r_base = rag_query_base(q, max_new_tokens=200)
    r_lora = rag_query_lora(q, max_new_tokens=200)

    print(f"\n{'─' * 70}")
    print(f"Q: {q}")
    print(f"\n  [BASE] {r_base['answer'][:200]}...")
    print(f"\n  [LORA] {r_lora['answer'][:200]}...")
    print(f"\n  Tiempo base: {r_base['timing']} | lora: {r_lora['timing']}")


In [ ]:
import gradio as gr

def chat_fn(message, history, use_lora=True):
    if use_lora:
        result = rag_query_lora(message, k=5, max_new_tokens=400)
    else:
        result = rag_query_base(message, k=5, max_new_tokens=400)

    sources_md = "### Fuentes recuperadas\n\n"
    for i, s in enumerate(result['sources'], 1):
        m = s['metadata']
        sources_md += (
            f"**[{i}] {m.get('institution', '?')}** ({m.get('year', '?')})\n"
            f"_{m.get('source_file', '?')}_ — pág. {m.get('pages', '?')}\n\n"
            f"> {s['content'][:300]}\n\n"
        )
    sources_md += f"\n*Tiempos: {result['timing']}*"
    return result['answer'], sources_md


with gr.Blocks(title="RAG Seguridad México", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# RAG Seguridad Pública México\n"
        "Chat sobre el corpus de 23 PDFs institucionales. "
        "Cada respuesta muestra las 5 fuentes con institución, año, archivo y página."
    )
    with gr.Row():
        use_lora = gr.Checkbox(label="Usar LoRA fine-tuned", value=True)
    chatbot = gr.Chatbot(label="Conversación", height=400)
    with gr.Row():
        msg = gr.Textbox(
            label="Tu pregunta",
            placeholder="¿Cuántos homicidios hubo en 2024?",
            scale=4,
        )
        send = gr.Button("Enviar", scale=1)
    sources_out = gr.Markdown(label="Fuentes")

    def respond(message, chat_history, use_lora):
        if not message.strip():
            return "", chat_history, ""
        answer, sources = chat_fn(message, chat_history, use_lora)
        chat_history = chat_history + [(message, answer)]
        return "", chat_history, sources

    msg.submit(respond, [msg, chatbot, use_lora], [msg, chatbot, sources_out])
    send.click(respond, [msg, chatbot, use_lora], [msg, chatbot, sources_out])

    gr.Examples(
        examples=[
            "¿Cuántos homicidios hubo en México en 2024?",
            "¿Quién es el autor del corpus?",
            "¿Qué instituciones están representadas?",
            "¿Tiene el corpus una postura editorial?",
            "¿Cuál es la cifra negra de criminalidad?",
            "¿Qué dice INEGI sobre desplazamientos forzados?",
            "¿Cuáles son las críticas a la militarización?",
        ],
        inputs=msg,
    )

demo.launch(share=True, debug=True)

print("\n✓ Interfaz Gradio lanzada. El link 'public URL' expira en 72 hs.")


In [ ]:
import os

print("=" * 60)
print("VERIFICACIÓN FINAL")
print("=" * 60)

sizes_gb = {
    "corpus (PDFs)": sum(f.stat().st_size for f in CORPUS_DIR.rglob("*") if f.is_file()) / 1e9,
    "vectordb": sum(f.stat().st_size for f in VECTORDB_DIR.rglob("*") if f.is_file()) / 1e9,
    "chunks.jsonl": CHUNKS_FILE.stat().st_size / 1e6,
    "lora adapter": sum(f.stat().st_size for f in LORA_DIR.rglob("*") if f.is_file()) / 1e6,
    "hf cache": sum(f.stat().st_size for f in HF_CACHE.rglob("*") if f.is_file()) / 1e9,
}

print("\nEspacio en Drive:")
for name, size in sizes_gb.items():
    unit = "GB" if size > 1 else "MB"
    val = size if size > 1 else size * 1000
    print(f"  {name:<20}: {val:.1f} {unit}")

total = sum(sizes_gb.values())
print(f"\n  TOTAL: {total:.2f} GB")

print("\n--- Checklist ---")
checks = [
    ("GPU T4 detectada", torch.cuda.is_available()),
    ("23 PDFs en corpus", len(list(CORPUS_DIR.rglob('*.pdf'))) >= 20),
    ("chunks.jsonl existe", CHUNKS_FILE.exists()),
    ("vectordb existe", (VECTORDB_DIR / "chroma.sqlite3").exists()),
    ("LoRA adapter entrenado", (LORA_DIR / "adapter_config.json").exists()),
    ("Modelo + LoRA en GPU", next(lora_model.parameters()).device.type == "cuda"),
]
for name, ok in checks:
    print(f"  [{'✓' if ok else '✗'}] {name}")

print("\n--- Próxima sesión ---")
print("1. Abrir el notebook")
print("2. Reejecutar celda 3 (mount) + celda 5 (paths)")
print("3. Saltar directo a celda 11 (cargar modelo)")
print("4. Saltar a celda 15 (cargar LoRA)")
print("5. Saltar a celda 18 (lanzar Gradio)")
print("Tiempo total: ~2 min para tener el chat funcionando")
